# SFT results

What supervised finetuning does to a base model, run by run. Every number here
comes from the run's own `artifacts/logs/*.jsonl`, so a cell re-reads the
archive instead of re-training -- re-running this notebook costs seconds, not
GPU hours.

A new finetuning run appends a section. LoRA and DPO get their own notebooks.

The task interface is `task.py`; the loop is `sft.py`; the run is one line:

    python basic.py sft --task reverse --steps 1000 --batch-size 8 --grad-accum 2


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import math
import sys
from pathlib import Path

root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.append(str(root / "src"))
sys.path.append(str(root / "src" / "video"))

import torch

from paths import CKPT_DIR, LOG_DIR

DEV = "cuda" if torch.cuda.is_available() else "cpu"

# label -> (log stem, training window). sft.py logs the window from 2026-09-21
# on; these three runs predate that line, so it is recorded by hand.
RUNS = {
    "scratch 11.2M": ("sft_smoke_2026-09-21_21-45-49", 128),
    "smoke base 30M": ("sft_reverse_2026-09-21_22-13-20", 256),
    "fineweb 123.6M": ("sft_reverse_2026-09-21_22-16-03", 256),
}
CKPT = CKPT_DIR / "sft_reverse_2026-09-21_22-16-03.pt"


def load_run(label):
    """(config, eval rows, summary) -- the jsonl a Run wrote is the archive."""
    stem, window = RUNS[label]
    lines = (LOG_DIR / f"{stem}.jsonl").read_text().splitlines()
    events = [json.loads(line) for line in lines]
    pick = lambda e: [x for x in events if x["event"] == e]
    return pick("config")[0] | {"window": window}, pick("log"), pick("summary")[0]


def table(rows, cols, w=12):
    print("".join(f"{c:>{w}}" for c in cols))
    for r in rows:
        print(
            "".join(
                f"{v:>{w}.4f}"
                if isinstance(v, float)
                else f"{'-' if v is None else v:>{w}}"
                for v in (r.get(c) for c in cols)
            )
        )

## 1. does SFT solve reverse, and how fast?

The 123.6M FineWeb-Edu base, finetuned on `Reverse` with flex + document masking.
`exact_match` is the scoreboard: a greedy decode of 200 fresh words, scored by
string equality. **Reads a log, < 1 s.**

Recorded result: **1.000 exact match by step 250**, 1000 steps in 4.6 min on a 4060.

| step | comp  | prompt | exact_match |
| ---- | ----- | ------ | ----------- |
| 0    | 6.067 | 8.728  | 0.000       |
| 50   | 0.432 | 8.019  | 0.400       |
| 100  | 0.007 | 8.797  | 0.995       |
| 250  | 0.001 | 9.059  | **1.000**   |
| 1000 | 0.000 | 9.816  | 1.000       |

Completion loss starts at **6.07, not ln(50259) = 10.8** — that gap is what
pretraining bought on a task the base has never seen. Everything past step 250 is
the prompt loss drifting; 300 steps is the honest setting.


In [3]:
cfg, rows, summ = load_run("fineweb 123.6M")
print(
    f"{cfg['n_embed']}d x {cfg['n_layer']}L   window {cfg['window']}   "
    f"batch {cfg['batch_size']}x{cfg['grad_accum_steps']}   lr {cfg['lr']}   "
    f"{cfg['attention']}"
)
table(rows, ["step", "comp", "prompt", "exact_match", "reward", "stop_rate"])
print(
    f"\nbest exact_match {summ['best_exact_match']:.3f}   "
    f"{summ['total_time_s']:.0f}s total"
)

768d x 12L   window 256   batch 8x2   lr 3e-05   flex
        step        comp      prompt exact_match      reward   stop_rate
           0      6.0672      8.7281      0.0000      0.0061      0.0000
          50      0.4321      8.0188      0.4000      0.7365      0.9650
         100      0.0074      8.7969      0.9950      0.9994      1.0000
         150      0.0055      9.1750      0.9900      0.9975      0.9900
         200      0.0018      8.9437      0.9950      0.9994      1.0000
         250      0.0005      9.0594      1.0000      1.0000      1.0000
         300      0.0001      8.8781      1.0000      1.0000      1.0000
         350      0.0004      8.7688      1.0000      1.0000      1.0000
         400      0.0091      8.6031      0.9900      0.9979      1.0000
         450      0.0003      8.7719      1.0000      1.0000      1.0000
         500      0.0045      9.2781      0.9950      0.9994      0.9950
         550      0.0001      9.6687      1.0000      1.0000      1.00

## 2. what masking costs the prompt

`comp` is the loss on completion tokens (trained on). `prompt` is the loss on
prompt tokens, which are masked out of the gradient and only watched. It does not
rise monotonically — it **falls, then climbs past where it started**. **< 1 s.**

Recorded result: from scratch **10.8 → 6.5 → 11.7**; from the pretrained base
**8.73 → 8.02 → 9.82**.

| run            | start | dip       | end   |
| -------------- | ----- | --------- | ----- |
| scratch 11.2M  | 10.82 | 6.48 @100 | 11.70 |
| fineweb 123.6M | 8.73  | 8.02 @50  | 9.82  |

Two legs. Down: training on completions teaches _this data only uses 27 tokens_,
which helps every position including the untrained ones. Up: the model
specialises into a reversal machine and becomes **confidently wrong** about
prompt tokens.

The dip is **0.7 nats from the pretrained base against 4.3 from scratch** — the
base already models text, so there is far less free alphabet-learning to collect.
Neither run reaches the floor: prompt letters are uniform over 26, so ln(26) =
3.26 is the best any model can do there.


In [4]:
for label in ("scratch 11.2M", "fineweb 123.6M"):
    cfg, rows, _ = load_run(label)
    dip = min(rows, key=lambda r: r["prompt"])
    print(
        f"{label}   start {rows[0]['prompt']:.2f} -> dip {dip['prompt']:.2f} "
        f"@{dip['step']} -> end {rows[-1]['prompt']:.2f}"
        f"   (fell {rows[0]['prompt'] - dip['prompt']:.2f} nats first)"
    )
    table(rows, ["step", "comp", "prompt", "exact_match"])
    print()
print(f"floor for a random prompt letter: ln(26) = {math.log(26):.2f}")

scratch 11.2M   start 10.81 -> dip 6.48 @100 -> end 11.70   (fell 4.33 nats first)
        step        comp      prompt exact_match
           0     10.8125     10.8125      0.0000
         100      0.3391      6.4797      0.6500
         200      0.0050      9.7500      1.0000
         300      0.0020     11.0437      1.0000
         400      0.0005     11.7031      1.0000

fineweb 123.6M   start 8.73 -> dip 8.02 @50 -> end 9.82   (fell 0.71 nats first)
        step        comp      prompt exact_match
           0      6.0672      8.7281      0.0000
          50      0.4321      8.0188      0.4000
         100      0.0074      8.7969      0.9950
         150      0.0055      9.1750      0.9900
         200      0.0018      8.9437      0.9950
         250      0.0005      9.0594      1.0000
         300      0.0001      8.8781      1.0000
         350      0.0004      8.7688      1.0000
         400      0.0091      8.6031      0.9900
         450      0.0003      8.7719      1.0000
  

## 3. what the model predicts where it was never trained

Feed a prompt and read the top prediction at each prompt position — positions the
loss never scored. **Loads the checkpoint, ~10 s.**

Recorded result: it collapses onto **an early letter of the word**, at p = 0.8–0.98 — though the first position can be soft.

| prompt   | predictions at prompt positions                   |
| -------- | ------------------------------------------------- |
| `dog>`   | `d`(0.82) `d`(0.97) `d`(0.77)                     |
| `quiz>`  | `q`(0.45) `q`(0.97) `q`(0.92) `u`(0.82)           |
| `zebra>` | `d`(0.22) `z`(0.87) `e`(0.78) `e`(0.93) `e`(0.98) |

Not a clean rule, and not the same failure the small from-scratch model has — that
one echoes _the current token_ at p = 1.00 (`c`→`c`, `a`→`a`, `t`→`t`). The
obvious guess is attention dumping mass on position 0, the sink. **Untested**:
flex and SDPA do not hand back attention weights, so this is a hypothesis, not a
measurement.

This is what the 9.82 prompt loss above looks like from the inside — the
model has not forgotten the alphabet (mass on the 26 letters stays ~1.0), it has
become certain.


In [5]:
from checkpoint import load_checkpoint
from reverse import Reverse
from tokenizer import ENDOFTEXT

task = Reverse()  # the constructor only builds the char table, not the data
tok = task.tok
model, _ = load_checkpoint(CKPT, DEV)
model.eval()
letters = torch.tensor([int(i) for i in task.char_ids], device=DEV)
d = lambda i: tok.decode([int(i)]).replace(ENDOFTEXT, "<eot>")

for word in ["cat", "dog", "quiz", "zebra"]:
    seq = [tok.encode(c)[0] for c in word] + [task.sep_id]
    with torch.no_grad():
        p = model(torch.tensor([seq], device=DEV))[0].float().softmax(-1)
    top = "  ".join(f"{d(p[t].argmax())!r}({p[t].max():.2f})" for t in range(len(word)))
    mass = p[: len(word)][:, letters].sum(-1).min()
    print(f"{word + '>':<8} {top:<46} min letter-mass {mass:.3f}")

cat>     'c'(0.63)  'c'(0.94)  'c'(0.88)                min letter-mass 0.850
dog>     'd'(0.82)  'd'(0.97)  'd'(0.77)                min letter-mass 0.878
quiz>    'q'(0.45)  'q'(0.97)  'q'(0.92)  'u'(0.82)     min letter-mass 0.840
zebra>   'd'(0.22)  'z'(0.87)  'e'(0.78)  'e'(0.93)  'e'(0.98) min letter-mass 0.798


## 4. what a run costs

Three runs on the same task, same 4060. **< 1 s.**

Recorded result: the 123.6M base solves reverse at **lr 3e-5** in ~100 steps;
from scratch needs **lr 1e-3**, and 200 steps.

| run            | base           | lr   | steps to 1.000           | total |
| -------------- | -------------- | ---- | ------------------------ | ----- |
| scratch 11.2M  | none           | 1e-3 | 200                      | 22 s  |
| smoke base 30M | fineweb_smoke  | 3e-5 | not reached (0.655 @400) | 49 s  |
| fineweb 123.6M | fineweb 123.6M | 3e-5 | 250                      | 276 s |

The 30M row is an accident worth keeping: `latest_ckpt("fineweb")` prefix-matched
`fineweb_smoke_*` and finetuned the wrong model. Its base was trained on a
different corpus, so its val loss is not comparable to the 123.6M one — which is
exactly how the bug hid.

**The memory ceiling is the logits, not the weights.** Batch 16 × window 256 OOMs
on 8 GB: `[16, 256, 50259]` in fp32 is ~785 MiB, and the backward wants another.
Halving the batch and accumulating keeps tokens-per-step identical.


In [6]:
print(
    f"{'run':>16}{'window':>8}{'tok/step':>10}{'steps':>7}{'s/step':>9}"
    f"{'total s':>9}{'exact':>8}"
)
for label in RUNS:
    cfg, rows, summ = load_run(label)
    per_step = cfg["batch_size"] * cfg["grad_accum_steps"] * cfg["window"]
    print(
        f"{label:>16}{cfg['window']:>8}{per_step:>10}{cfg['max_steps']:>7}"
        f"{summ['total_time_s'] / cfg['max_steps']:>9.2f}"
        f"{summ['total_time_s']:>9.0f}{summ['best_exact_match']:>8.3f}"
    )

print(
    f"\npeak logits at batch 16, window 256: "
    f"{16 * 256 * 50259 * 4 / 2**20:.0f} MiB in fp32"
)

             run  window  tok/step  steps   s/step  total s   exact
   scratch 11.2M     128      4096    400     0.06       24   1.000
  smoke base 30M     256      4096    400     0.12       49   0.655
  fineweb 123.6M     256      4096   1000     0.28      276   1.000

peak logits at batch 16, window 256: 785 MiB in fp32
